In [ ]:
# OUTPUT TABLES
# 1. dim_match.csv
# 2. dim_player.csv
# 3. fact_shots.csv
# 4. fact_set_pieces.csv
# 5. fact_set_piece_events.csv

from pathlib import Path
import numpy as np
import pandas as pd

TARGET_TEAM = "Gil Vicente FC"

DATA_FOLDER = Path(    r"C:\Users\francisco.frota\Downloads\dados\dados" )

OUTPUT_FOLDER = Path(    r"C:\Users\francisco.frota\Downloads\gil_vicente_challenge_1\powerbi_data" )

OUTPUT_FOLDER.mkdir(parents=True,exist_ok=True)

PITCH_LENGTH = 105.0
FREE_KICK_MIN_X = 63.0
FINAL_THIRD_MIN_X = 70.0

files = sorted(DATA_FOLDER.glob("*.pkl"))

if not files:
    raise FileNotFoundError(f"No PKL files found in {DATA_FOLDER}")

frames = []

for file in files:
    temp = pd.read_pickle(file).copy()
    temp["source_file"] = file.stem
    frames.append(temp)

df = pd.concat(frames,ignore_index=True)

print(f"Loaded {len(files)} files | "f"{len(df):,} rows | "f"{df.shape[1]} columns")

# ================================================================
# 3. HELPERS
# ================================================================

def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).upper().strip()


def first_not_null(series):
    values = series.dropna()
    if values.empty: 
        return np.nan 
    return values.iloc[0]

def safe_sum(series):
    return float(pd.to_numeric(series,errors="coerce").fillna(0).sum())

def numeric_value(row, col):
    return pd.to_numeric(row.get(col),errors="coerce")

# ================================================================
# 4. NUMERIC CLEANING
# ================================================================

numeric_cols = [    "playerId","pressingPlayerId","passReceiverPlayerId","setPieceId","setPiecePhaseIndex","setPieceSubPhaseId","setPieceSubPhaseIndex",
    "sequenceIndex","eventNumber","periodId", "gameTimeInSec", "startAdjCoordinatesX","startAdjCoordinatesY","endAdjCoordinatesX",
    "endAdjCoordinatesY","SHOT_XG","POSTSHOT_XG","SHOT_AT_GOAL_NUMBER","SHOT_AT_GOAL_OFF_TARGET_NUMBER","SHOT_AT_GOAL_NUMBER_ON_TARGET",
    "SHOT_AT_GOAL_NUMBER_SUCCESS","SHOT_AT_GOAL_NUMBER_SAVED","SHOT_AT_GOAL_NUMBER_CAUGHT","SHOT_AT_GOAL_NUMBER_BLOCKED","SHOT_AT_GOAL_NUMBER_OTHER",
    "PXT_ATTACK","PXT_SETPIECE","PXT_PASS","PXT_DRIBBLE","PXT_REC","PXT_SHOT","BYPASSED_OPPONENTS","BYPASSED_DEFENDERS","GOALS","OWNGOALS"]

for col in numeric_cols: if col in df.columns: df[col] = pd.to_numeric(df[col],errors="coerce")

df = (df.sort_values(["matchId","eventNumber"]).reset_index(drop=True))

# ================================================================
# 5. COORDINATES
# ================================================================

df["pitch_start_x"] = (df["startAdjCoordinatesX"]+ 52.5)

df["pitch_start_y"] = (df["startAdjCoordinatesY"]+ 34)

df["pitch_end_x"] = (df["endAdjCoordinatesX"]+ 52.5)

df["pitch_end_y"] = (df["endAdjCoordinatesY"]+ 34)

# ================================================================
# 6. MATCH DIMENSION
# ================================================================

dim_match = (df.groupby("matchId").agg(date=("dateTime", "first"), competition=("competitionName", "first"), home_team=("homeSquadName", "first"),away_team=("awaySquadName", "first")).reset_index())
dim_match["date"] = pd.to_datetime(dim_match["date"], errors="coerce")
dim_match["opponent"] = np.where(dim_match["home_team"] == TARGET_TEAM,dim_match["away_team"],dim_match["home_team"])
dim_match["venue"] = np.where(dim_match["home_team"] == TARGET_TEAM,"Home","Away")
dim_match = (dim_match.sort_values("date").reset_index(drop=True))
dim_match["match_order"] = (np.arange(1,len(dim_match) + 1))
dim_match["match_label"] = (dim_match["opponent"] + " ("+ dim_match["venue"].str[0] + ")")
match_opponent_map = dict(zip(dim_match["matchId"],dim_match["opponent"]))
match_venue_map = dict(zip(dim_match["matchId"],dim_match["venue"]))
df["opponent"] = (df["matchId"].map(match_opponent_map))
df["venue"] = (df["matchId"].map(match_venue_map))

# ================================================================
# 7. PLAYER DIMENSION
# ================================================================

player_identity = (df[["playerId","playerName","squadName"]].dropna(subset=["playerId","playerName"]).drop_duplicates())
name_counts = (player_identity.groupby("playerName")["playerId"].nunique())
duplicate_names = set(name_counts[name_counts >1].index)

def build_player_label(player_id, player_name):
    if pd.isna(player_name):
        return None
    if (player_name in duplicate_names and pd.notna(player_id)):
        return (f"{player_name} "f"[{int(player_id)}]")
    return str(player_name)


df["playerLabel"] = [build_player_label(player_id,player_name) for player_id, player_name in zip(df["playerId"],df["playerName"])]
dim_player = (df[["playerId","playerName","playerLabel","squadName","playerPosition","playerPositionSide"]].dropna(subset=["playerId"]).drop_duplicates(subset=["playerId"]))

# ================================================================
# 8. SET-PIECE-LINKED EVENTS
# ================================================================

sp_linked_events = (df[df["setPieceId"].notna()].copy())

if sp_linked_events.empty: raise ValueError( "No rows with setPieceId found.")

# Validate ID uniqueness across matches
sp_id_check = (sp_linked_events.groupby("setPieceId")["matchId"].nunique())

if (sp_id_check > 1).any(): raise ValueError("setPieceId is not unique across matches.")

# ================================================================
# 9. RESTART ROW
# ================================================================

RESTART_ACTION_TYPES = {"CORNER","FREE_KICK","THROW_IN","GOAL_KICK","KICK_OFF"}

def choose_restart_row(group):
    group = (group.sort_values("eventNumber"))
    # penalty override
    penalty = group[group["action"].astype(str).str.upper().str.strip().eq("PENALTY_KICK")]
    if not penalty.empty:
        return penalty.iloc[0]
    restart = group[group["actionType"].astype(str).str.upper().isin(RESTART_ACTION_TYPES)]
    if not restart.empty:
        return restart.iloc[0]
    return group.iloc[0]

# ================================================================
# 10. SET-PIECE TYPE
# ================================================================

def infer_set_piece_type(row):
    action = clean_text(row.get("action"))
    action_type = clean_text(row.get("actionType"))
    # explicit penalty always wins
    if (action == "PENALTY_KICK" or action_type == "PENALTY_KICK"):
        return "Penalty"
    mapping = {"CORNER": "Corner", "FREE_KICK": "Free kick","THROW_IN": "Throw-in", "GOAL_KICK": "Goal kick","KICK_OFF": "Kick-off"}
    if action_type in mapping:
        return mapping[action_type]
    text = " ".join([clean_text(row.get("setPieceCategory")), clean_text(row.get("adjSetPieceCategory")), clean_text(row.get("setPieceExecutionType")), action])
    if "PENALTY" in text:
        return "Penalty"
    if "CORNER" in text:
        return "Corner"
    if ("THROW_IN" in text or "THROW IN" in text):
        return "Throw-in"
    if ("FREE_KICK" in text or "FREE KICK" in text):
        return "Free kick"
    if ("GOAL_KICK" in text or "GOAL KICK" in text):
        return "Goal kick"
    if ("KICK_OFF" in text or "KICK OFF" in text or "KICKOFF" in text):
        return "Kick-off"
    return "Other"

# ================================================================
# 11. ANALYTICAL SET-PIECE SCOPE
# ================================================================

def is_final_third_throw_in(set_piece_type,start_pitch_position,start_x):
    if set_piece_type != "Throw-in":
        return False
    pitch_position = clean_text(start_pitch_position)
    if pitch_position in {"FINAL_THIRD","OPPONENT_BOX"}:
        return True
    return (pd.notna(start_x) and start_x >= FINAL_THIRD_MIN_X)

def classify_analysis_sp_type(set_piece_type,start_pitch_position, start_x):
    if set_piece_type == "Corner":
        return "Corner"
    if ( set_piece_type == "Free kick" and pd.notna(start_x) and start_x >= FREE_KICK_MIN_X):
        return "Attacking free kick"
    if is_final_third_throw_in(set_piece_type,start_pitch_position,start_x):
        return "Attacking throw-in"
    if set_piece_type == "Penalty":
        return "Penalty"
    return "Other set piece"

# ================================================================
# 12. ONE ROW PER SET PIECE
# ================================================================

set_piece_records = []

for set_piece_id, linked_group in (sp_linked_events.groupby("setPieceId")):
    linked_group = (linked_group.sort_values("eventNumber"))
    restart = choose_restart_row(linked_group)
    match_id = first_not_null(linked_group["matchId"])
    attacking_team = first_not_null(linked_group["attackingSquadName"])
    side = ("For" if attacking_team == TARGET_TEAM else "Against")
    first_event = linked_group.iloc[0]
    attacking_event_mask = (linked_group["squadName"] == attacking_team)
    n_attacking_events = int(attacking_event_mask.sum())
    set_piece_type = (infer_set_piece_type(restart))
    start_x = restart.get("pitch_start_x")
    start_y = restart.get("pitch_start_y")
    start_pitch_position = (restart.get("startPitchPosition"))
    analysis_sp_type = (classify_analysis_sp_type(set_piece_type,start_pitch_position,start_x))
    in_scope_attacking_sp = (analysis_sp_type in {"Corner","Attacking free kick","Attacking throw-in"})
    native_group = (linked_group[linked_group["phase"] == "SET_PIECE"].copy())
    if native_group.empty: native_group = (linked_group.copy())
    shot_rows = (native_group[native_group["actionType"] == "SHOT"].copy())
    shot_rows = shot_rows[~shot_rows["action"].astype(str).str.upper().str.strip().eq("PENALTY_KICK")]
    shots = (shot_rows["eventId"].nunique())
    xg = (safe_sum(shot_rows["SHOT_XG"]) if "SHOT_XG" in shot_rows.columns else 0.0)
    goals = (safe_sum(shot_rows["SHOT_AT_GOAL_NUMBER_SUCCESS"]) if "SHOT_AT_GOAL_NUMBER_SUCCESS" in shot_rows.columns else 0.0)
    taker_id = restart.get("playerId")
    taker_name = restart.get("playerName")
    record = {
        "setPieceId": set_piece_id,
        "matchId":match_id,
        "opponent": match_opponent_map.get(match_id),
        "venue": match_venue_map.get(match_id),
        "periodId": restart.get("periodId"),
        "gameTime": restart.get("gameTime"),
        "gameTimeInSec":restart.get("gameTimeInSec"),
        "restart_eventNumber": restart.get("eventNumber"),
        "side": side,
        "attacking_team": attacking_team,
        "set_piece_type":set_piece_type,
        "analysis_sp_type": analysis_sp_type,
        "in_scope_attacking_sp": in_scope_attacking_sp,
        "taker_id": taker_id,
        "taker_name": taker_name,
        "taker_label":build_player_label(taker_id,taker_name),
        "restart_actionType": restart.get("actionType"),
        "restart_action":restart.get("action"),
        "start_x":start_x,
        "start_y":start_y,
        "start_pitch_position":start_pitch_position,
        "start_lane":restart.get("startLane"),
        "end_x":first_event.get("pitch_end_x"),
        "end_y":first_event.get("pitch_end_y"),
        "end_pitch_position":first_event.get("endPitchPosition"),
        "end_lane":first_event.get("endLane"),
        "end_zone":first_event.get("endPackingZone"),
        "n_attacking_events":n_attacking_events,
        "distance_from_goal_line": (PITCH_LENGTH - start_x  if pd.notna(start_x)  else np.nan ),
        "set_piece_category":first_not_null(native_group["setPieceCategory"]) if "setPieceCategory" in native_group.columns else np.nan,
        "adj_set_piece_category": first_not_null(native_group["adjSetPieceCategory"]) if "adjSetPieceCategory" in native_group.columns else np.nan,
        "execution_type": first_not_null(native_group["setPieceExecutionType"]) if "setPieceExecutionType" in native_group.columns else np.nan,
        "start_zone": first_not_null(native_group["setPieceSubPhaseStartZone"]) if "setPieceSubPhaseStartZone" in native_group.columns else np.nan,
        "shots": int(shots),
        "xg": float(xg),
        "goals": float(goals),
        "shot_created":bool(shots > 0),
        "pxt_attack":(safe_sum(native_group["PXT_ATTACK"]) if "PXT_ATTACK" in native_group.columns else 0.0),
        "pxt_setpiece":(safe_sum(native_group["PXT_SETPIECE"]) if "PXT_SETPIECE" in native_group.columns else np.nan),
        "corner_type": first_not_null(native_group["setPieceSubPhaseCornerType"])if "setPieceSubPhaseCornerType" in native_group.columns else np.nan,
        "ball_trajectory": first_not_null(native_group["setPieceSubPhaseBallTrajectory"])if "setPieceSubPhaseBallTrajectory" in native_group.columns else np.nan,
        "main_event": first_not_null(native_group["setPieceSubPhaseMainEvent"])if "setPieceSubPhaseMainEvent" in native_group.columns else np.nan,
        "main_event_player_id":first_not_null(native_group["setPieceSubPhaseMainEventPlayerId"]) if "setPieceSubPhaseMainEventPlayerId" in native_group.columns else np.nan,
        "main_event_player_name": first_not_null(native_group["setPieceSubPhaseMainEventPlayerName"]) if "setPieceSubPhaseMainEventPlayerName" in native_group.columns else np.nan,
        "main_event_outcome": first_not_null(native_group["setPieceSubPhaseMainEventOutcome"]) if "setPieceSubPhaseMainEventOutcome" in native_group.columns else np.nan,
        "first_touch_player_id": first_not_null(native_group["setPieceSubPhaseFirstTouchPlayerId"]) if "setPieceSubPhaseFirstTouchPlayerId" in native_group.columns else np.nan,
        "first_touch_player_name": first_not_null(native_group["setPieceSubPhaseFirstTouchPlayerName"]) if "setPieceSubPhaseFirstTouchPlayerName" in native_group.columns else np.nan,
        "first_touch_won": first_not_null(native_group["setPieceSubPhaseFirstTouchWon"]) if "setPieceSubPhaseFirstTouchWon" in native_group.columns else np.nan,
        "second_touch_player_id": first_not_null(native_group["setPieceSubPhaseSecondTouchPlayerId"]) if "setPieceSubPhaseSecondTouchPlayerId" in native_group.columns else np.nan,
        "second_touch_player_name": first_not_null(native_group["setPieceSubPhaseSecondTouchPlayerName"]) if "setPieceSubPhaseSecondTouchPlayerName" in native_group.columns else np.nan,
        "second_touch_won": first_not_null(native_group["setPieceSubPhaseSecondTouchWon"]) if "setPieceSubPhaseSecondTouchWon" in native_group.columns else np.nan,
        "second_touch_end_zone": first_not_null(native_group["setPieceSubPhaseSecondTouchEndZone"]) if "setPieceSubPhaseSecondTouchEndZone" in native_group.columns
            else np.nan
    }
    set_piece_records.append(record)

fact_set_pieces = pd.DataFrame(set_piece_records)

# ================================================================
# 13. SHOT OUTCOME
# ================================================================

def classify_shot_outcome(row):
    success = numeric_value(row,"SHOT_AT_GOAL_NUMBER_SUCCESS")
    caught = numeric_value(row,"SHOT_AT_GOAL_NUMBER_CAUGHT")
    saved = numeric_value(row,"SHOT_AT_GOAL_NUMBER_SAVED")
    blocked = numeric_value(row,"SHOT_AT_GOAL_NUMBER_BLOCKED")
    off_target = numeric_value(row,"SHOT_AT_GOAL_OFF_TARGET_NUMBER")
    other = numeric_value(row,"SHOT_AT_GOAL_NUMBER_OTHER")
    on_target = numeric_value(row,"SHOT_AT_GOAL_NUMBER_ON_TARGET")
    if pd.notna(success) and success > 0:
        return "Goal"
    if pd.notna(caught) and caught > 0:
        return "Caught"
    if pd.notna(saved) and saved > 0:
        return "Saved"
    if pd.notna(blocked) and blocked > 0:
        return "Blocked"
    if pd.notna(off_target) and off_target > 0:
        return "Off target"
    if pd.notna(other) and other > 0:
        return "Other"
    if pd.notna(on_target) and on_target > 0:
        return "On target - unspecified"
    return "Unknown"

# ================================================================
# 14. ALL NON-PENALTY SHOTS
# ================================================================

shots = (df[df["actionType"] == "SHOT"].copy())
shots = (shots[~shots["action"].astype(str).str.upper().str.strip().eq("PENALTY_KICK")].copy())
shots["side"] = np.where(shots["squadName"] == TARGET_TEAM,"For","Against")

sp_lookup = (
    fact_set_pieces[[
            "setPieceId",
            "set_piece_type",
            "analysis_sp_type",
            "in_scope_attacking_sp",
            "start_x",
            "start_y",
            "start_pitch_position"]].drop_duplicates(subset=["setPieceId"]))

shots = (shots.merge(sp_lookup,on="setPieceId",how="left",suffixes=("","_restart")))
shots = (shots[shots["set_piece_type"].isna()|shots["set_piece_type"].ne("Penalty")].copy())

# ================================================================
# 15. SHOT PLAY PATTERN
# ================================================================

def classify_shot_play_pattern(row):
    phase = clean_text(row.get("phase"))
    if phase == "IN_POSSESSION":
        return "In possession"
    if phase == "ATTACKING_TRANSITION":
        return "Attacking transition"
    if phase == "SECOND_BALL":
        return "Second ball"
    if phase == "SET_PIECE":
        if (row.get("set_piece_type") == "Corner"):
            return "Corner"
        if (row.get("analysis_sp_type") == "Attacking free kick" ):
            return "Attacking free kick"
        if (row.get("analysis_sp_type") == "Attacking throw-in"):
            return "Attacking throw-in"
        return "Other set piece"
    if phase:
        return (phase.replace("_", " ").title())
    return "Other / Unknown"

shots["play_pattern_analysis"] = (shots.apply(classify_shot_play_pattern,axis=1))
shots["shot_outcome"] = (shots.apply(classify_shot_outcome, axis=1))
shots["is_goal"] = (shots["shot_outcome"] == "Goal")
shots["is_on_target"] = (shots["shot_outcome"].isin(["Goal","Saved","Caught","On target - unspecified"]))
shots["is_blocked"] = (shots["shot_outcome"] == "Blocked")

# ================================================================
# 16. FACT SHOTS
# ================================================================

shot_columns = [
    "matchId","eventId","eventNumber","sequenceIndex","setPieceId","periodId","gameTime","gameTimeInSec","opponent","venue","side",
    "squadName","attackingSquadName","phase","play_pattern_analysis","set_piece_type","analysis_sp_type","in_scope_attacking_sp",
    "playerId","playerName","playerLabel","playerPosition","playerPositionSide","action","bodyPart","result","pitch_start_x",
    "pitch_start_y","startPitchPosition","startLane","distanceToGoal","shotDistance","shotAngle","shotTargetPointY","shotTargetPointZ",
    "shotWoodwork","shotGkAdjCoordinatesX","shotGkAdjCoordinatesY","shotGkDivePointY","shotGkDivePointZ","SHOT_XG","POSTSHOT_XG",
    "SHOT_AT_GOAL_NUMBER","SHOT_AT_GOAL_OFF_TARGET_NUMBER","SHOT_AT_GOAL_NUMBER_ON_TARGET","SHOT_AT_GOAL_NUMBER_SUCCESS",
    "SHOT_AT_GOAL_NUMBER_SAVED","SHOT_AT_GOAL_NUMBER_CAUGHT","SHOT_AT_GOAL_NUMBER_BLOCKED","SHOT_AT_GOAL_NUMBER_OTHER","shot_outcome","is_goal",
    "is_on_target","is_blocked","SHOT_XG_FROM_PASSES","SHOT_CREATING_ACTIONS","SHOT_ASSISTS","EXPECTED_SHOT_ASSISTS","EXPECTED_GOAL_ASSISTS","PXT_SHOT"
]

shot_columns = [col for col in shot_columns if col in shots.columns]
fact_shots = (shots[shot_columns].copy())
fact_shots = (fact_shots.rename(columns={
            "squadName":   "team",
            "SHOT_XG":     "xg",
            "POSTSHOT_XG": "postshot_xg",
            "pitch_start_x":"shot_x",
            "pitch_start_y":"shot_y"  }))
# ================================================================
# 17. FACT SET-PIECE EVENTS
# ================================================================

event_columns = [
    "matchId","setPieceId","eventId","eventNumber","sequenceIndex","periodId","gameTime","gameTimeInSec","opponent","venue",
    "phase","attackingSquadName","squadName","playerId","playerName","playerLabel","playerPosition","passReceiverPlayerId",
    "passReceiverPlayerName","actionType","action","bodyPart","result","pitch_start_x","pitch_start_y","pitch_end_x","pitch_end_y",
    "startPitchPosition","endPitchPosition","startLane","endLane","startPackingZone","endPackingZone","setPiecePhaseIndex",
    "setPieceSubPhaseId","setPieceSubPhaseIndex","setPieceCategory","adjSetPieceCategory","setPieceExecutionType",
    "setPieceSubPhaseStartZone","setPieceSubPhaseCornerEndZone","setPieceSubPhaseCornerType","setPieceSubPhaseBallTrajectory",
    "setPieceSubPhaseMainEvent","setPieceSubPhaseMainEventPlayerId","setPieceSubPhaseMainEventPlayerName","setPieceSubPhaseMainEventOutcome",
    "setPieceSubPhaseFirstTouchPlayerId","setPieceSubPhaseFirstTouchPlayerName","setPieceSubPhaseFirstTouchWon",
    "setPieceSubPhaseSecondTouchPlayerId","setPieceSubPhaseSecondTouchPlayerName","setPieceSubPhaseSecondTouchWon",
    "setPieceSubPhaseSecondTouchEndZone","SHOT_XG","POSTSHOT_XG","SHOT_AT_GOAL_NUMBER_SUCCESS","SHOT_AT_GOAL_NUMBER_SAVED",
    "SHOT_AT_GOAL_NUMBER_CAUGHT","SHOT_AT_GOAL_NUMBER_BLOCKED","SHOT_AT_GOAL_OFF_TARGET_NUMBER","PXT_ATTACK","PXT_SETPIECE",
    "PXT_PASS","PXT_DRIBBLE","PXT_REC","PXT_SHOT","BYPASSED_OPPONENTS","BYPASSED_DEFENDERS"
]

event_columns = [col for col in event_columns if col in sp_linked_events.columns]

fact_set_piece_events = (sp_linked_events[event_columns].copy())

# Add the restart-level tactical classification
event_lookup = (fact_set_pieces[["setPieceId","side","set_piece_type","analysis_sp_type","in_scope_attacking_sp","taker_id","taker_name",
        "taker_label","end_x","end_y","end_pitch_position","end_lane","end_zone","n_attacking_events"]].drop_duplicates(subset=["setPieceId"]))

fact_set_piece_events = (fact_set_piece_events.merge(event_lookup,on="setPieceId",how="left"))
fact_set_piece_events = (fact_set_piece_events.sort_values(["setPieceId","eventNumber"]).reset_index(drop=True))
fact_set_piece_events["sequence_event_order"] = (fact_set_piece_events.groupby("setPieceId").cumcount() + 1)
fact_set_piece_events["is_attacking_team_event"] = (fact_set_piece_events["squadName"] == fact_set_piece_events["attackingSquadName"])

# Order considering attacking-team events only. Opponent events remain NaN.
fact_set_piece_events["attacking_event_order"] = np.nan
attacking_mask = (fact_set_piece_events["is_attacking_team_event"])
fact_set_piece_events.loc[attacking_mask,"attacking_event_order"] = (fact_set_piece_events.loc[attacking_mask].groupby("setPieceId").cumcount() + 1)
fact_set_piece_events["is_native_set_piece_phase"] = (fact_set_piece_events["phase"] == "SET_PIECE")
fact_set_piece_events["is_penalty_action"] = (fact_set_piece_events["action"].astype(str).str.upper().str.strip().eq("PENALTY_KICK"))

# ================================================================
# 18. EXPORT
# ================================================================

def export_csv(frame,filename):
    path = (OUTPUT_FOLDER/ filename)
    frame.to_csv(path,index=False,encoding="utf-8-sig")
    print(f"{filename}: "f"{len(frame):,} rows")


export_csv(dim_match,"dim_match.csv")
export_csv(dim_player,"dim_player.csv")
export_csv(fact_shots,"fact_shots.csv")
export_csv(fact_set_pieces,"fact_set_pieces.csv")
export_csv(fact_set_piece_events, "fact_set_piece_events.csv")

print("Done")

In [ ]:
import matplotlib.pyplot as plt
from mplsoccer import VerticalPitch
from matplotlib.patches import FancyArrowPatch
from pathlib import Path

df = fact_set_pieces.copy()

OUTPUT_DIR = Path(
    r"C:\Users\francisco.frota\Downloads\gil_vicente_challenge\powerbi_data"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# QUICK CHECK OF THE DATA
# ============================================================

corners = df[
    (df["side"] == "For")
    & (df["set_piece_type"] == "Corner")
].copy()

print(f"Total Gil Vicente attacking corners: {len(corners)}")

print("\nCorners by side:")
print(corners["start_lane"].value_counts(dropna=False))

print("\nCorners by destination type:")
print(corners["corner_type"].value_counts(dropna=False))


# ============================================================
# DRAWING FUNCTION
# ============================================================

def draw_corner_zones(data,
    corner_side,
    save_path=None
):

    # --------------------------------------------------------
    # FILTER
    # --------------------------------------------------------

    plot_df = data[
        (data["side"] == "For")
        & (data["set_piece_type"] == "Corner")
        & (data["start_lane"] == corner_side)
    ].copy()

    # --------------------------------------------------------
    # COUNTS
    # --------------------------------------------------------

    near = int(
        (plot_df["corner_type"] == "CORNER_NEAR_POST").sum()
    )

    central = int(
        (plot_df["corner_type"] == "CORNER_CENTRAL").sum()
    )

    far = int(
        (plot_df["corner_type"] == "CORNER_FAR_POST").sum()
    )

    # Sum the less direct / alternative corner types
    other_types = [
        "CORNER_OPEN_PLAY",
        "CORNER_VARIANT",
        "CORNER_OTHER"
    ]

    other = int(
        plot_df["corner_type"].isin(other_types).sum()
    )

    total = len(plot_df)

    # --------------------------------------------------------
    # PRINT SUMMARY
    # --------------------------------------------------------

    print("\n" + "=" * 40)
    print(corner_side)
    print("=" * 40)
    print(f"Total corners: {total}")
    print(f"Near post:     {near}")
    print(f"Central:       {central}")
    print(f"Far post:      {far}")
    print(f"Other:         {other}")

    # Useful check:
    classified_total = near + central + far + other

    print(f"Classified:    {classified_total}")

    if classified_total != total:
        print(
            f"WARNING: {total - classified_total} corners "
            "are not included in the four plotted categories."
        )

    # --------------------------------------------------------
    # PITCH
    # --------------------------------------------------------

    pitch = VerticalPitch(
        pitch_type="statsbomb",
        half=True,
        goal_type="box",
        line_zorder=3
    )

    fig, ax = pitch.draw(
        figsize=(6, 6.5)
    )

    # --------------------------------------------------------
    # ZONE GEOMETRY
    # --------------------------------------------------------

    goal_line = 120

    # How deep target zones extend
    zone_bottom = 105

    # Central zone = between goal posts
    central_left = 36
    central_right = 44

    # Outer near/far limits
    outer_left = 18
    outer_right = 62

    # --------------------------------------------------------
    # DASHED ZONE SEPARATORS
    # --------------------------------------------------------

    ax.plot(
        [central_left, central_left],
        [zone_bottom, goal_line],
        linestyle="--",
        linewidth=1.5,
        zorder=4
    )

    ax.plot(
        [central_right, central_right],
        [zone_bottom, goal_line],
        linestyle="--",
        linewidth=1.5,
        zorder=4
    )

    # Bottom of the three main target zones
    ax.plot(
        [outer_left, outer_right],
        [zone_bottom, zone_bottom],
        linestyle="--",
        linewidth=1.2,
        zorder=4
    )

    # --------------------------------------------------------
    # MAIN ZONE CENTRES
    # --------------------------------------------------------

    left_zone_x = (
        outer_left + central_left
    ) / 2

    central_zone_x = (
        central_left + central_right
    ) / 2

    right_zone_x = (
        central_right + outer_right
    ) / 2

    # --------------------------------------------------------
    # MIRROR ACCORDING TO CORNER SIDE
    # --------------------------------------------------------

    if corner_side == "LEFT_WING":

        # From a LEFT corner:
        # left side = near post
        # right side = far post

        near_x = left_zone_x
        central_x = central_zone_x
        far_x = right_zone_x

        origin_x = 0

        # Alternative / short-style destination
        # outside penalty area, near the corner side
        other_x = 10
        other_y = 96

    elif corner_side == "RIGHT_WING":

        # From a RIGHT corner:
        # right side = near post
        # left side = far post

        far_x = left_zone_x
        central_x = central_zone_x
        near_x = right_zone_x

        origin_x = 80

        # Mirrored alternative destination
        other_x = 70
        other_y = 96

    else:
        raise ValueError(
            "corner_side must be LEFT_WING or RIGHT_WING"
        )

    origin_y = 120

    # --------------------------------------------------------
    # NUMBERS — MAIN ZONES
    # --------------------------------------------------------

    number_y = 111

    ax.text(
        near_x,
        number_y,
        str(near),
        ha="center",
        va="center",
        fontsize=26,
        fontweight="bold",
        zorder=6
    )

    ax.text(
        central_x,
        number_y,
        str(central),
        ha="center",
        va="center",
        fontsize=26,
        fontweight="bold",
        zorder=6
    )

    ax.text(
        far_x,
        number_y,
        str(far),
        ha="center",
        va="center",
        fontsize=26,
        fontweight="bold",
        zorder=6
    )

    # --------------------------------------------------------
    # NUMBER — OPEN PLAY + VARIANT + OTHER
    # --------------------------------------------------------

    ax.text(
        other_x,
        other_y,
        str(other),
        ha="center",
        va="center",
        fontsize=24,
        fontweight="bold",
        zorder=6
    )

    # --------------------------------------------------------
    # STRAIGHT ARROWS — MAIN ZONES
    # --------------------------------------------------------

    target_y = 106.5

    targets = [
        (near, near_x),
        (central, central_x),
        (far, far_x)
    ]

    for count, target_x in targets:

        if count == 0:
            continue

        if corner_side == "LEFT_WING":
            start_x = origin_x + 1
        else:
            start_x = origin_x - 1

        arrow = FancyArrowPatch(
            (start_x, origin_y - 0.5),
            (target_x, target_y),
            arrowstyle="-|>",
            mutation_scale=18,
            linewidth=1.7,
            connectionstyle="arc3,rad=0",
            zorder=5
        )

        ax.add_patch(arrow)

    # --------------------------------------------------------
    # SHORT ARROW — OTHER CORNER TYPES
    # --------------------------------------------------------

    if other > 0:

        if corner_side == "LEFT_WING":

            other_arrow_start = (
                origin_x + 1,
                origin_y - 1
            )

            other_arrow_end = (
                other_x,
                other_y + 3
            )

        else:

            other_arrow_start = (
                origin_x - 1,
                origin_y - 1
            )

            other_arrow_end = (
                other_x,
                other_y + 3
            )

        other_arrow = FancyArrowPatch(
            other_arrow_start,
            other_arrow_end,
            arrowstyle="-|>",
            mutation_scale=18,
            linewidth=1.7,
            connectionstyle="arc3,rad=0",
            zorder=5
        )

        ax.add_patch(other_arrow)

    # --------------------------------------------------------
    # LAYOUT
    # --------------------------------------------------------

    plt.tight_layout(pad=0.1)

    # --------------------------------------------------------
    # EXPORT
    # --------------------------------------------------------

    if save_path is not None:

        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
            transparent=True
        )

        print(f"Saved: {save_path}")

    # --------------------------------------------------------
    # SHOW
    # --------------------------------------------------------

    plt.show()

    return {
        "side": corner_side,
        "total": total,
        "near_post": near,
        "central": central,
        "far_post": far,
        "open_play_variant_other": other
    }


# ============================================================
# LEFT CORNERS
# ============================================================

left_summary = draw_corner_zones(
    data=fact_set_pieces,
    corner_side="LEFT_WING",
    save_path=OUTPUT_DIR / "gil_vicente_corners_left.png"
)


# ============================================================
# RIGHT CORNERS
# ============================================================

right_summary = draw_corner_zones(
    data=fact_set_pieces,
    corner_side="RIGHT_WING",
    save_path=OUTPUT_DIR / "gil_vicente_corners_right.png"
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\nLEFT:")
print(left_summary)

print("\nRIGHT:")
print(right_summary)

print("\nImages exported to:")
print(OUTPUT_DIR)